# 00 — One-year problem setup: orbital references and optical measurements

This notebook defines the **single-controller** simulation used by [01 — One-year NARX results](01_narx_results.ipynb). It replaces the legacy short-duration Gateway tracking comparisons.

**Controlled spacecraft:** the halo observer. Its corrected periodic Earth–Moon CR3BP orbit is the station-keeping reference; only this spacecraft receives commanded Δv.

**Tracking target:** Lunar Gateway. Its specified nominal 9:2 NRHO initial condition must pass numerical periodic closure in the same CR3BP; the resulting one-period trajectory is repeated for 365 days. This assumption represents a *prescribed maintained trajectory*, without including Gateway station-keeping commands or modeling disturbances beyond CR3BP.

**Navigation:** three lunar surface observatories (LSOs) provide optical angles for observer EKF updates, with observer-to-Gateway camera angles for Gateway tracking when visible. The controller uses estimated observer states and causal maneuver histories, never simulated truth.

This notebook verifies the trajectory references and illustrates one-year *nominal visibility geometry*. It does not claim tracking accuracy from geometry alone.

## State, reference, and measurement models

The normalized rotating Earth–Moon CR3BP state is $\boldsymbol{x}=[\boldsymbol{r},\boldsymbol{v}]^\top$ with $\dot{\boldsymbol x}=f(\boldsymbol x)$.

A periodic observer reference satisfies $\boldsymbol x_H^*(T_H)=\boldsymbol x_H^*(0)$. A supplied Gateway seed is accepted as periodic only when its one-cycle position and velocity closure residuals meet the configured strict numerical tolerances.

An accepted LSO optical observation is modeled as

$$
\boldsymbol z_{i,k}=h_i(\boldsymbol x_{H,k})+\boldsymbol\nu_{i,k},\qquad
\boldsymbol\nu_{i,k}\sim \mathcal N(0,R_i).
$$

The EKF estimates $\hat{\boldsymbol x}_{H,k},P_{H,k}$ from measurements; it is the **estimate**, not the true state, that informs STM and learned station-keeping decisions. Realized visibility depends on each policy's observer trajectory and is recomputed independently under common raw noise draws.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from halo_tracking.study import load_config
from halo_tracking.orbit_reference import correct_halo_reference
from halo_tracking.gateway_reference_periodic import gateway_periodic_reference
from halo_tracking.narx_dataset import _reference
from halo_tracking.network import sensor_configs
from halo_tracking.closed_loop_narx import _sample_observations

cfg = load_config()
corrected = correct_halo_reference(cfg)
T_H = float(corrected["orbit"]["period_tu"])
T_G = float(cfg["gateway"]["period_tu"])
x_H0 = np.asarray(corrected["orbit"]["state"], dtype=float)
observer_one_cycle = _reference(x_H0, T_H, cfg)
def observer_at(t):
    return np.asarray(observer_one_cycle(float(t) % T_H), dtype=float)

gateway_at, gateway_check = gateway_periodic_reference(cfg)
observer_check = corrected["orbit"]["correction"]
display(Markdown(
    f"**Observer periodic closure:** {observer_check['position_closure_km']:.3e} km; "
    f"{observer_check['velocity_closure_m_s']:.3e} m/s.  "
    f"**Gateway periodic closure:** {gateway_check['position_closure_km']:.3e} km; "
    f"{gateway_check['velocity_closure_m_s']:.3e} m/s.  "
    f"Gateway period: {gateway_check['period_days']:.4f} days."
))

## One-year nominal visibility geometry

This is a **geometry check**, not a time history of EKF estimation error. Optical acceptance accounts for the repository's elevation and exclusion masks. The controlled trajectories in Notebook 01 recalculate acceptance using their actual propagated observer states.

In [ ]:
sites = sensor_configs(cfg)
rng = np.random.default_rng(1022)
days = np.arange(366, dtype=float)
tus = days * 86400 / cfg["time_unit_s"]
lso_visible = np.zeros((len(days), len(sites)), dtype=bool)
gateway_visible = np.zeros(len(days), dtype=bool)
range_km = np.zeros(len(days))
for k, t in enumerate(tus):
    obs = observer_at(t)
    target = gateway_at(t)
    _, lso_visible[k], _, gateway_visible[k] = _sample_observations(
        t, obs, target, cfg, rng, sites)
    range_km[k] = np.linalg.norm(target[:3] - obs[:3]) * cfg["length_unit_km"]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for j, site in enumerate(cfg["sensors"]):
    ax[0].step(days, lso_visible[:, j].astype(float)+j*1.25, where="mid",
               label=site["name"])
ax[0].set_yticks([])
ax[0].set_ylabel("LSO availability (offset)")
ax[0].legend(loc="upper right", ncol=3, fontsize=8)
ax[1].plot(days, range_km, label="Observer–Gateway range (km)")
ax[1].set_ylabel("Range (km)")
ax[1].set_xlabel("Elapsed days")
ax[1].grid(alpha=.25)
fig.suptitle("365-day nominal CR3BP visibility and relative geometry")
plt.tight_layout()
plt.show()
display(Markdown(
    f"**Nominal visibility:** at least one LSO accepted on "
    f"{100*np.mean(lso_visible.any(axis=1)):.1f}% of sampled days; "
    f"observer–Gateway optical link accepted on "
    f"{100*np.mean(gateway_visible):.1f}% of sampled days. "
    "This is not a guarantee of estimation accuracy."
))

## Continue to the one-year controller comparison

Open [Notebook 01 — One-year NARX results](01_narx_results.ipynb) for measured-state STM-teacher training, actual 365-day uncontrolled/STM/NARX executions, and observer-departure, Δv, and Gateway-tracking plots. The results notebook executes a one-arc year-long pilot by default and can be configured for additional paired Monte Carlo arcs.